# Stage 3: paired scene bootstrap for the 15 updates from old C1 seed 1

In [ ]:
import os, sys, glob, json, time, tarfile, shutil
from pathlib import Path
BASE_SEED = 1
T0 = time.time()
def clock(m): print(f'[{time.time()-T0:7.1f}s] {m}', flush=True)
ROOT = Path('/kaggle/working/E1_new'); ROOT.mkdir(parents=True, exist_ok=True)
# saved models + test export (campaign inputs), Kaggle may have unpacked the .tar
src = None
for p in glob.glob('/kaggle/input/**/models/stage1b/manifest.json', recursive=True):
    src = Path(p).parent.parent.parent; break
if src is None:
    tar = glob.glob('/kaggle/input/**/campaign_inputs.tar', recursive=True); assert tar, 'campaign inputs missing'
    with tarfile.open(tar[0]) as tf: tf.extractall(ROOT)
else:
    for sub in ['data', 'models']: shutil.copytree(src / sub, ROOT / sub, dirs_exist_ok=True)
# latest mlpipe code (with bootstrap.py)
code = glob.glob('/kaggle/input/**/mlpipe/bootstrap.py', recursive=True)
if code:
    shutil.copytree(Path(code[0]).parent, ROOT / 'mlpipe', dirs_exist_ok=True)
else:
    tar = glob.glob('/kaggle/input/**/mlpipe_code.tar', recursive=True); assert tar, 'mlpipe code missing'
    with tarfile.open(tar[0]) as tf: tf.extractall(ROOT)
assert (ROOT / 'mlpipe/bootstrap.py').exists() and (ROOT / 'data/stage1/singapore_test.json').exists()
DET = sorted(glob.glob('/kaggle/input/**/detections_upd_b%d_*.json' % BASE_SEED, recursive=True))
assert len(DET) == 15, f'expected 15 detection files, found {len(DET)}'
sys.path.insert(0, str(ROOT))
clock(f'inputs ready: base seed {BASE_SEED}, {len(DET)} updates')

In [ ]:
from mlpipe import data as D
from mlpipe.campaign import Evaluator
from mlpipe.bootstrap import scene_sums, paired_bootstrap
EXPORT = json.load(open(ROOT / 'data/stage1/singapore_test.json'))
for f in DET:
    EXPORT['detections'][Path(f).stem.replace('detections_', '')] = json.load(open(f))
ev = Evaluator(EXPORT, ROOT / 'models/stage1b', BASE_SEED, log=clock)
out = {'base': ev.base, 'n_boot': 2000, 'method': 'paired scene bootstrap, 50 test scenes, 95% interval', 'updates': {}}
names = [Path(f).stem.replace('detections_', '') for f in DET] + [ev.other]
for setting in ('old_pipeline', 'isolated'):
    base = scene_sums(ev, setting, ev.base)
    for n in names:
        r = paired_bootstrap(base, scene_sums(ev, setting, n))
        out['updates'].setdefault(n, {})[setting] = r
        clock(f"{setting:12s} {n:22s} d2 {r['delta2_minADE']['point']:+.3f} [{r['delta2_minADE']['ci95'][0]:+.3f},{r['delta2_minADE']['ci95'][1]:+.3f}] "
              f"d3 {r['delta3_L2@3s']['point']:+.3f} [{r['delta3_L2@3s']['ci95'][0]:+.3f},{r['delta3_L2@3s']['ci95'][1]:+.3f}]")
json.dump(out, open(f'/kaggle/working/bootstrap_base_s{BASE_SEED}.json', 'w'), indent=1)
for setting in ('old_pipeline', 'isolated'):
    ups = [n for n in names if n != ev.other]
    h2 = sum(out['updates'][n][setting]['delta2_minADE']['harm'] for n in ups)
    h3 = sum(out['updates'][n][setting]['delta3_L2@3s']['harm'] for n in ups)
    clock(f'{setting}: significant harm in prediction {h2}/15, planning {h3}/15')
shutil.rmtree(ROOT, ignore_errors=True)
clock('=== BOOTSTRAP DONE ===')